In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import trange
from tqdm import tqdm
import time
import re

In [2]:
headers = {"User-Agent": "Mozilla/5.0"}
base = "https://www.politifact.com/factchecks/list/?page="
root = "https://www.politifact.com"
data = []

In [3]:
def make_full_url(url):
    """Ensure PolitiFact URLs are absolute."""
    if url and url.startswith("/"):
        return root + url
    return url

In [4]:
def split_date_context(text):
    """
    Split strings like:
      'stated on November 9, 2025 in a Truth Social post'
    into clean date ('November 9, 2025') and context ('Truth Social post').
    Removes leading 'in', 'at', 'a', 'an' or ':'.
    """
    if not isinstance(text, str):
        return "", ""

    # Normalize spacing
    text = text.strip().replace(" ", " ")

    # Try common PolitiFact phrasing
    match = re.search(r"stated on ([A-Za-z]+\s+\d{1,2},\s*\d{4})(?:\s+(?:in|at)\s+(.*))?", text)
    date = match.group(1).strip()
    context = match.group(2).strip() if match.group(2) else ""

  
    # Clean leading prepositions or colons
    context = re.sub(r"^(in|at|on|a|an|:)\s+", "", context, flags=re.IGNORECASE)
    context = context.strip(" :")
    return date, context


In [5]:
def get_article_text(url):
    """Extract justification text (short + full) and metadata from a fact-check detail page."""
    try:
        url = make_full_url(url)
        r = requests.get(url, headers=headers, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        # --- Short justification (If Your Time is short) ---
        short_points = [li.get_text(strip=True) for li in soup.select(".m-callout.m-callout--large .short-on-time li p")]
        short_summary = " ".join(short_points).strip()

        # --- Full article text ---
        full_paras = [p.get_text(strip=True) for p in soup.select(".m-textblock p")]
        full_justification = " ".join(full_paras).strip()

        # --- Metadata section ---
        meta = {}
        for li in soup.select(".m-fact__meta li"):
            text = li.get_text(strip=True)
            if ":" in text:
                key, val = text.split(":", 1)
                meta[key.strip().lower()] = val.strip()

        justification = short_summary if short_summary else full_justification
        return justification, meta

    except Exception as e:
        print(f"⚠️ Failed to parse article {url}: {e}")
        return "", {}

In [6]:
for page in trange(1, 20):
    url = base + str(page)
    try:
        r = requests.get(url, headers=headers, timeout=10)
        r.raise_for_status()
    except Exception as e:
        print(f"⚠️ Skipping page {page}: {e}")
        continue

    soup = BeautifulSoup(r.text, "html.parser")
    cards = soup.select(".m-statement")

    for card in cards:
        quote_tag = card.select_one(".m-statement__quote")
        link_tag = card.select_one(".m-statement__quote a")
        rating_tag = card.select_one(".m-statement__meter img")
        speaker_tag = card.select_one(".m-statement__meta a")
        date_tag = card.select_one(".m-statement__desc")

        # Skip incomplete cards
        if not (quote_tag and link_tag and rating_tag):
            continue

        quote = quote_tag.get_text(strip=True)
        link = make_full_url(link_tag["href"]) if link_tag else None
        rating = rating_tag["alt"].strip() if rating_tag else None
        speaker = speaker_tag.get_text(strip=True) if speaker_tag else None
        date_raw = date_tag.get_text(strip=True) if date_tag else ""
        justification, meta = get_article_text(link)

        date, context = split_date_context(date_raw)

        data.append({
            "speaker": speaker,
            "quote": quote,
            "rating": rating,
            "date": date,
            "context": context,
            "justification": justification,
            "link": link
        })

    time.sleep(2)  # politeness delay

100%|██████████| 19/19 [03:22<00:00, 10.64s/it]


In [7]:
current_statement = pd.DataFrame(data)
current_statement.head()

,speaker,quote,rating,date,context,justification,link
0,Donald Trump,“DOGE halts yearly payment of $2.5 million to ...,pants-fire,"November 9, 2025",Truth Social post,The claim originated from a satirical article....,https://www.politifact.com/factchecks/2025/nov...
1,Anderson Clayton,“North Carolina teachers are already the lowes...,barely-true,"October 29, 2025",press release,North Carolina Democrats are calling on state ...,https://www.politifact.com/factchecks/2025/nov...
2,X posts,“RFK Jr. flees the scene after Novo Nordisk Ex...,false,"November 6, 2025",X post,Gordon Findlay is a Novo Nordisk manager based...,https://www.politifact.com/factchecks/2025/nov...
3,Nancy Mace,New York City Mayor-elect Zohran Mamdani “is b...,pants-fire,"November 6, 2025",fundraising email,Zohran Mamdani has never said he seeks to impl...,https://www.politifact.com/factchecks/2025/nov...
4,Donald Trump,Voting in California is “rigged.”,pants-fire,"November 4, 2025",Truth Social post,"As evidence, the White House said California’s...",https://www.politifact.com/factchecks/2025/nov...


In [8]:
def get_speaker_info(url):
    """Extract name, description, metadata, and scorecard history."""
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # --- Speaker name ---
    name_tag = soup.select_one(".m-person__title") or soup.select_one(".m-pageheader__title")
    if not name_tag:
        raise ValueError("Not a valid person page")
    name = name_tag.get_text(strip=True)

    # --- Description ---
    desc_tag = soup.select_one(".m-pageheader__body p")
    description = desc_tag.get_text(strip=True) if desc_tag else ""

    # --- Metadata (party, state, etc.) ---
    meta_items = {}
    for li in soup.select(".m-person__info li"):
        key_tag, val_tag = li.find("span"), li.find("div")
        if key_tag and val_tag:
            key = key_tag.get_text(strip=True).rstrip(":")
            val = val_tag.get_text(strip=True)
            meta_items[key] = val

    # --- Scorecard (speaker's truth history) ---
    scorecard_items = soup.select(".m-scorecard__item")
    for item in scorecard_items:
        label_tag = item.select_one(".m-scorecard__title")
        pct_tag = item.select_one("[data-scorecard-value]")
        checks_tag = item.select_one(".m-scorecard__checks a")

        if not label_tag:
            continue

        label = label_tag.get_text(strip=True).split("\n")[0].strip()
        pct = pct_tag.get_text(strip=True) if pct_tag else "0"
        checks = checks_tag.get_text(strip=True).split()[0] if checks_tag else "0"

        clean = label.replace(" ", "_").lower()
        meta_items[f"history_{clean}_pct"] = pct
        meta_items[f"history_{clean}_checks"] = checks

    return name, description, meta_items



In [9]:
def format_speaker_url(speaker_name):
    """Convert 'Joe Biden' → https://www.politifact.com/personalities/joe-biden/"""
    slug = speaker_name.lower().replace(" ", "-")
    return f"https://www.politifact.com/personalities/{slug}/"

In [10]:
speaker_history = []
unique_speakers = current_statement["speaker"].dropna().unique()

for sp in tqdm(unique_speakers):
    try:
        url = format_speaker_url(sp)
        name, desc, meta = get_speaker_info(url)
        meta.update({"speaker": name, "description": desc, "url": url})
        speaker_history.append(meta)
    except ValueError:
        print(f"⚠️ Skipping {sp}: not a valid person page")
    except Exception as e:
        print(f"⚠️ Error scraping {sp}: {e}")
    time.sleep(2)

  2%|▏         | 2/116 [00:04<04:11,  2.21s/it]

⚠️ Skipping X posts: not a valid person page


  9%|▉         | 11/116 [00:25<04:04,  2.33s/it]

⚠️ Skipping The Ronald Reagan Presidential Foundation and Institute: not a valid person page


 18%|█▊        | 21/116 [00:49<03:49,  2.42s/it]

...

 83%|████████▎ | 96/116 [03:51<00:48,  2.43s/it]

⚠️ Skipping Kevin O'Leary: not a valid person page


 97%|█████████▋| 112/116 [04:30<00:09,  2.37s/it]

⚠️ Skipping Donald Trump Jr.: not a valid person page


100%|██████████| 116/116 [04:40<00:00,  2.42s/it]


In [11]:
def extract_party(desc):
    """Detect political affiliation keywords in a speaker's description."""
    if not isinstance(desc, str):
        return None
    text = desc.lower()

    # --- direct keyword matches ---
    if re.search(r"\b(democrat(ic)?|democratic[-\s]?farmer[-\s]?labor|dfl)\b", text):
        return "Democrat"
    if re.search(r"\b(republican|gop)\b", text):
        return "Republican"
    if re.search(r"\b(independent|no[-\s]?party)\b", text):
        return "Independent"
    if re.search(r"\b(libertarian)\b", text):
        return "Libertarian"
    if re.search(r"\b(green\s?party|green)\b", text):
        return "Green"
    if re.search(r"\b(constitution\s?party)\b", text):
        return "Constitution"
    if re.search(r"\b(tea\s?party)\b", text):
        return "Tea Party"
    if re.search(r"\b(liberal\s?party)\b", text):
        return "Liberal"
    if re.search(r"\b(moderate)\b", text):
        return "Moderate"

    # --- shorthand like (D), (R), (I) ---
    if re.search(r"\(d\)", text):
        return "Democrat"
    if re.search(r"\(r\)", text):
        return "Republican"
    if re.search(r"\(i\)", text):
        return "Independent"

    return "Unknown"


In [12]:
speaker_history_df = pd.DataFrame(speaker_history)
check_cols = [c for c in speaker_history_df.columns if c.endswith("_checks")]

# Keep only speaker, description, and the count columns
speaker_history_df = speaker_history_df[["speaker", "description"] + check_cols]

# Rename columns
speaker_history_df.columns = [
    col.replace("history_", "").replace("_checks", "_count") for col in speaker_history_df.columns
]

speaker_history_df["party_affiliation"] = speaker_history_df["description"].apply(extract_party)

cols = ["speaker", "party_affiliation"] + [
    c for c in speaker_history_df.columns if c.endswith("_count")
]
speaker_history_df = speaker_history_df[cols]

In [13]:
speaker_history_df

,speaker,party_affiliation,true_count,mostly_true_count,half_true_count,mostly_false_count,false_count,pants_on_fire_count
0,Donald Trump,Unknown,36,86,129,215,441,219
1,Anderson Clayton,Democrat,0,0,0,1,1,0
2,Nancy Mace,Republican,0,0,1,2,1,1
3,Elon Musk,Unknown,0,0,0,0,9,1
4,JB Pritzker,Unknown,1,5,3,7,4,1
...,...,...,...,...,...,...,...,...
101,Robin Vos,Unknown,3,9,4,6,7,1
102,Joe Biden,Democrat,27,77,79,60,65,7
103,Dana Loesch,Unknown,0,0,1,2,2,0
104,Tony Evers,Democrat,8,10,8,10,7,1


In [14]:
current_statement

,speaker,quote,rating,date,context,justification,link
0,Donald Trump,“DOGE halts yearly payment of $2.5 million to ...,pants-fire,"November 9, 2025",Truth Social post,The claim originated from a satirical article....,https://www.politifact.com/factchecks/2025/nov...
1,Anderson Clayton,“North Carolina teachers are already the lowes...,barely-true,"October 29, 2025",press release,North Carolina Democrats are calling on state ...,https://www.politifact.com/factchecks/2025/nov...
2,X posts,“RFK Jr. flees the scene after Novo Nordisk Ex...,false,"November 6, 2025",X post,Gordon Findlay is a Novo Nordisk manager based...,https://www.politifact.com/factchecks/2025/nov...
3,Nancy Mace,New York City Mayor-elect Zohran Mamdani “is b...,pants-fire,"November 6, 2025",fundraising email,Zohran Mamdani has never said he seeks to impl...,https://www.politifact.com/factchecks/2025/nov...
4,Donald Trump,Voting in California is “rigged.”,pants-fire,"November 4, 2025",Truth Social post,"As evidence, the White House said California’s...",https://www.politifact.com/factchecks/2025/nov...
...,...,...,...,...,...,...,...
565,Facebook posts,"Says Elon Musk said, “The Boeing Starliner’s s...",false,"January 4, 2025",Facebook post,This claim is unfounded. NASA said in December...,https://www.politifact.com/factchecks/2025/jan...
566,Threads posts,“Mark Cuban shifts company from Texas to Calif...,false,"January 2, 2025",Threads post,This claim originated on a self-described sati...,https://www.politifact.com/factchecks/2025/jan...
567,Threads posts,"After Donald Trump offered to buy Greenland, “...",false,"December 25, 2024",Threads post,The claim that Denmark offered to buy the U.S....,https://www.politifact.com/factchecks/2025/jan...
568,Facebook posts,“DEAL DONE: New Orleans Saints owner Gayle Ben...,false,"January 2, 2025",post on Facebook,We found no credible reports or information as...,https://www.politifact.com/factchecks/2025/jan...


In [ ]:
merged_df = current_statement.merge(
    speaker_history_df,
    on="speaker",        # join on speaker name
    how="left"           # keep all statements, even if no speaker history
)
·

In [17]:
merged_df.to_csv('polifact.csv')

In [2]:
import pandas as pd

In [4]:
pd.read_csv("polifact.csv").head()

,Unnamed: 0,speaker,quote,rating,date,context,justification,link,party_affiliation,true_count,mostly_true_count,half_true_count,mostly_false_count,false_count,pants_on_fire_count
0,0,Donald Trump,“DOGE halts yearly payment of $2.5 million to ...,pants-fire,"November 9, 2025",Truth Social post,The claim originated from a satirical article....,https://www.politifact.com/factchecks/2025/nov...,Unknown,36.0,86.0,129.0,215.0,441.0,219.0
1,1,Anderson Clayton,“North Carolina teachers are already the lowes...,barely-true,"October 29, 2025",press release,North Carolina Democrats are calling on state ...,https://www.politifact.com/factchecks/2025/nov...,Democrat,0.0,0.0,0.0,1.0,1.0,0.0
2,2,X posts,“RFK Jr. flees the scene after Novo Nordisk Ex...,false,"November 6, 2025",X post,Gordon Findlay is a Novo Nordisk manager based...,https://www.politifact.com/factchecks/2025/nov...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,Nancy Mace,New York City Mayor-elect Zohran Mamdani “is b...,pants-fire,"November 6, 2025",fundraising email,Zohran Mamdani has never said he seeks to impl...,https://www.politifact.com/factchecks/2025/nov...,Republican,0.0,0.0,1.0,2.0,1.0,1.0
4,4,Donald Trump,Voting in California is “rigged.”,pants-fire,"November 4, 2025",Truth Social post,"As evidence, the White House said California’s...",https://www.politifact.com/factchecks/2025/nov...,Unknown,36.0,86.0,129.0,215.0,441.0,219.0
